# proteon-graphein quickstart

Interactive walkthrough of [proteon-graphein](https://github.com/theGreatHerrLebert/proteon-graphein) —
the Graphein adapter that attaches `proteon` structural features to a
[Graphein](https://github.com/a-r-j/graphein) `nx.Graph`.

Covers:

1. Build a residue-level Graphein graph from a PDB
2. Attach proteon features (SASA, RSA, DSSP, energy, hbond_count, dihedrals)
3. Inspect node-level and graph-level attributes
4. Switch to atom-level granularity (Graphein's `granularity="atom"`)
5. Many graphs in one parallel call
6. Energy-only mode (graph-level features without touching nodes)

Requires: `proteon>=0.2.0`, `graphein`, `networkx`. Sibling project
[proteon-pyg](https://github.com/theGreatHerrLebert/proteon-pyg) does
the same job for users who want PyG `Data` tensors instead.


In [1]:
from pathlib import Path
import urllib.request

import graphein.protein as gp
from graphein.protein.config import ProteinGraphConfig

import proteon_graphein

print(f'proteon_graphein {proteon_graphein.__version__}')


def get_pdb(pdb_id: str) -> Path:
    """Resolve a PDB fixture by id ('1crn'). Looks for a repo-local
    ``test-pdbs/`` first, then falls back to caching a download from
    RCSB into ``~/.cache/proteon-pyg/pdbs/``. Pure stdlib -- no extra
    dependencies for someone running this notebook fresh."""
    name = f'{pdb_id.lower()}.pdb'
    repo_local = Path.cwd() / 'test-pdbs' / name
    if repo_local.exists():
        return repo_local
    cache = Path.home() / '.cache' / 'proteon-pyg' / 'pdbs'
    cache.mkdir(parents=True, exist_ok=True)
    cached = cache / name
    if not cached.exists():
        url = f'https://files.rcsb.org/download/{pdb_id.upper()}.pdb'
        print(f'downloading {pdb_id.upper()} from RCSB...')
        urllib.request.urlretrieve(url, cached)
    return cached


PDB_1CRN = get_pdb('1crn')
PDB_1AKE = get_pdb('1ake')


proteon_graphein 0.2.0


## 1. Residue-level graph + feature attach

The default Graphein `ProteinGraphConfig()` builds a residue-level graph
(one node per residue, CA position). `add_proteon_features` mutates the
graph in place and returns it.


In [2]:
graph = gp.construct_graph(config=ProteinGraphConfig(), path=str(PDB_1CRN))
print(f'graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges')

graph = proteon_graphein.add_proteon_features(
    graph, PDB_1CRN,
    sasa=True, dssp=True, energy=True,
    hbond_count=True, dihedrals=True,
)

# Look at one node to see what got attached
sample_id, sample = next(iter(graph.nodes(data=True)))
print(f'\nnode {sample_id!r} attrs:')
for k, v in sample.items():
    print(f'  {k}: {v!r}')


Output()

graph: 46 nodes, 45 edges

node 'A:THR:1' attrs:
  chain_id: 'A'
  residue_name: 'THR'
  residue_number: 1
  atom_type: 'CA'
  element_symbol: 'C'
  coords: array([16.967, 12.784,  4.338], dtype=float32)
  b_factor: 10.800000190734863
  meiler: dim_1    3.03
dim_2    0.11
dim_3    2.60
dim_4    0.26
dim_5    5.60
dim_6    0.21
dim_7    0.36
Name: THR, dtype: float64
  residue_sasa: 71.36573164287555
  rsa: 0.41491704443532296
  dssp: 'C'
  hbond_count: 1
  psi: 147.6600264500666


In [3]:
# Graph-level energy attaches once per graph
print(f'energy components: {sorted(graph.graph["proteon_energy"])}')
print(f'total energy:      {graph.graph["proteon_energy"]["total"]:.2f} kJ/mol')
print(f'force field:       {graph.graph["proteon_ff"]}')


energy components: ['angle_bend', 'bond_stretch', 'electrostatic', 'improper_torsion', 'n_14_pairs', 'n_angles', 'n_bonds', 'n_excluded_pairs', 'n_impropers', 'n_topo_atoms', 'n_torsions', 'n_unassigned_atoms', 'solvation', 'torsion', 'total', 'vdw']
total energy:      93.19 kJ/mol
force field:       charmm19_eef1


NaN positions are *absent* on the node, not attached as `float('nan')` —
this is a deliberate contract:


In [4]:
n_with_phi = sum(1 for _, d in graph.nodes(data=True) if 'phi' in d)
n_with_psi = sum(1 for _, d in graph.nodes(data=True) if 'psi' in d)
print(f'nodes with phi: {n_with_phi}/{graph.number_of_nodes()} (N-terminus skipped)')
print(f'nodes with psi: {n_with_psi}/{graph.number_of_nodes()} (C-terminus skipped)')


nodes with phi: 45/46 (N-terminus skipped)
nodes with psi: 45/46 (C-terminus skipped)


## 2. Atom-level graph

Graphein supports atom granularity. proteon-graphein auto-detects it
from `graph.graph["config"].granularity` and attaches per-atom features
(`atom_sasa`, `charge`, `is_backbone`, `hetero`) **plus** broadcast
residue features so each atom node knows its residue's DSSP / SASA / etc.


In [5]:
atom_config = ProteinGraphConfig(granularity='atom')
g_atom = gp.construct_graph(config=atom_config, path=str(PDB_1CRN))
g_atom = proteon_graphein.add_proteon_features(
    g_atom, PDB_1CRN, hbond_count=True, dihedrals=True
)

print(f'atom graph: {g_atom.number_of_nodes()} nodes')

sample_id, sample = next(iter(g_atom.nodes(data=True)))
print(f'\nnode {sample_id!r} attrs:')
for k in sorted(sample):
    print(f'  {k}: {sample[k]!r}')


Output()

atom graph: 327 nodes

node 'A:THR:1:N' attrs:
  atom_sasa: 20.390867092282775
  atom_type: 'N'
  b_factor: 13.789999961853027
  chain_id: 'A'
  charge: 0.0
  coords: array([17.047, 14.099,  3.625], dtype=float32)
  dssp: 'C'
  element_symbol: 'N'
  hbond_count: 1
  hetero: False
  is_backbone: True
  meiler: dim_1    3.03
dim_2    0.11
dim_3    2.60
dim_4    0.26
dim_5    5.60
dim_6    0.21
dim_7    0.36
Name: THR, dtype: float64
  psi: 147.6600264500666
  residue_name: 'THR'
  residue_number: 1
  residue_sasa: 71.36573164287555
  rsa: 0.41491704443532296


## 3. Mismatched-PDB safety

The adapter raises if the graph and PDB are misaligned. This is the
load-bearing check that catches the most common user mistake.


In [6]:
import networkx as nx

# Hand-build a graph whose node keys won't match anything in 1crn
fake_graph = nx.Graph()
fake_graph.add_node('Z:XXX:9999', chain_id='Z', residue_number=9999, residue_name='XXX')

try:
    proteon_graphein.add_proteon_features(fake_graph, PDB_1CRN, energy=False)
except ValueError as e:
    print(f'caught: {e}')


caught: No graph nodes matched proteon residues by (chain_id, residue_number, insertion_code). Was the graph built from the same PDB?


## 4. Many graphs in one parallel call

`add_proteon_features_batch` loads all PDBs and computes features in
parallel via proteon's batch primitives, then attaches per graph. Mixed
granularities in one batch are allowed.


In [7]:
g1 = gp.construct_graph(config=ProteinGraphConfig(), path=str(PDB_1CRN))
g2 = gp.construct_graph(config=ProteinGraphConfig(), path=str(PDB_1AKE))
g3 = gp.construct_graph(config=ProteinGraphConfig(granularity='atom'), path=str(PDB_1CRN))

paths = [PDB_1CRN, PDB_1AKE, PDB_1CRN]
graphs = proteon_graphein.add_proteon_features_batch(
    [g1, g2, g3], paths, hbond_count=True, dihedrals=True
)

for g, p in zip(graphs, paths):
    n = g.number_of_nodes()
    e = g.graph['proteon_energy']['total']
    granularity = g.graph['config'].granularity
    print(f'{p.name} ({granularity!r:>8}): {n:4} nodes, energy={e:8.2f} kJ/mol')


Output()

Output()

Output()

1crn.pdb (    'CA'):   46 nodes, energy=   93.19 kJ/mol
1ake.pdb (    'CA'):  428 nodes, energy=-1985.57 kJ/mol
1crn.pdb (  'atom'):  327 nodes, energy=   93.19 kJ/mol


## 5. Energy-only mode

If you only need graph-level energy and don't want to walk every node,
disable the per-node features. The "no nodes matched" guard is silenced
because no node-level features were requested.


In [8]:
g = gp.construct_graph(config=ProteinGraphConfig(), path=str(PDB_1CRN))
g = proteon_graphein.add_proteon_features(
    g, PDB_1CRN, sasa=False, dssp=False, energy=True
)

assert 'proteon_energy' in g.graph
sample_attrs = next(iter(g.nodes(data=True)))[1]
node_keys = set(sample_attrs)
proteon_keys = {'residue_sasa', 'rsa', 'dssp', 'hbond_count'}
print(f'no proteon node-level keys leaked: {not (node_keys & proteon_keys)}')
print(f'graph-level energy still attached:  {"proteon_energy" in g.graph}')


Output()

no proteon node-level keys leaked: True
graph-level energy still attached:  True


## What to try next

- Read [`evident.yaml`](../evident.yaml) for the parity claim contract —
  every value attached to every node is exact-equal to direct proteon
  API output, with NaN-aware skip semantics.
- Try [proteon-pyg](https://github.com/theGreatHerrLebert/proteon-pyg)
  for a PyG-tensor flavor of the same trust property.
- Build your own GNN pipeline. Two real conversion paths to a PyG
  `Data` if you want to train on these graphs:
  - `torch_geometric.utils.from_networkx(graph)` — PyG's first-party
    converter that preserves node/graph attributes that are scalars or
    vectors. (Watch out for non-tensor-friendly attrs like the `dssp`
    string; either drop them or convert before calling.)
  - [proteon-pyg](https://github.com/theGreatHerrLebert/proteon-pyg)
    bypasses Graphein entirely and emits the same proteon features
    straight onto a `Data` object. Cleanest path for PyG users.
